In [ ]:
SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
RUN_KEYS = ['polypythias__410m__seed-1__step-143000', 'polypythias__410m__seed-2__step-143000', 'polypythias__410m__seed-3__step-143000', 'polypythias__410m__seed-4__step-143000', 'polypythias__410m__seed-5__step-143000', 'polypythias__410m__seed-6__step-143000', 'polypythias__410m__seed-7__step-143000', 'polypythias__410m__seed-8__step-143000']
import json, pathlib, gzip
!git clone -q https://github.com/garyzhang1006/SNAP.git /tmp/SNAP && cd /tmp/SNAP && git checkout -q 6cfedee87ac52695f96665ecc0a62ecdc97e1c35
!pip install -q transformers==4.57.1
%cd /tmp/SNAP/compute2
!git log -1 --format=%h
jp = pathlib.Path('config/jobs.json')
d = json.loads(jp.read_text())
[j['filter'].update(seed=SEEDS, run_key=RUN_KEYS) for j in d['jobs'] if j['name'] == 'pythia_bank2_final']
jp.write_text(json.dumps(d, indent=1))
print([j['filter'] for j in json.loads(jp.read_text())['jobs'] if j['name'] == 'pythia_bank2_final'])
!python requests/build_bank2.py && python requests/freeze.py
fz = json.load(open('config/frozen.json'))['banks']
# No bank-two hash was frozen before scoring; the paper states 6,808 items, so a different count stops the run.
assert fz['bank2']['items'] == 6808, fz['bank2']
print('bank2 sha256', fz['bank2']['sha256'])
!mkdir -p /kaggle/working/scores && cp config/frozen.json config/jobs.json /kaggle/working/ && cp data/bank2_summary.json /kaggle/working/ 2>/dev/null; ls data
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python score_runs.py --job pythia_bank2_final --out /kaggle/working/scores --tmp /tmp/snap_tmp --deadline-hours 11.3
!ls /kaggle/working/scores/pythia_bank2_final
